# Fase 1.1 — Diagnóstico de listings de Airbnb en Río de Janeiro

**Objetivo:** comprobar la carga y conocer la calidad de los datos antes de definir reglas de limpieza.

Origen de lectura: `proy_airbnb_rio.staging.listings`. Este notebook no crea, modifica ni elimina tablas. Las conversiones se calculan en memoria como diagnóstico; no reemplazan los valores originales. Los únicos archivos que escribe son informes agregados en `reports/diagnostico/`.

## Cómo ejecutarlo

1. Guarda este archivo en la carpeta `notebooks/` de tu proyecto y abre la carpeta completa del proyecto en VS Code.
2. En **Seleccionar kernel**, elige el Python de tu `.venv` (3.11.9).
3. Comprueba que PostgreSQL esté iniciado. Ejecuta cada celda con **Shift + Enter**, en orden.
4. Revisa la configuración de conexión. Introduce la contraseña únicamente cuando aparezca la solicitud.
5. Si una celda falla, corrige ese error antes de continuar. Al terminar, comparte `resumen_diagnostico.json` y los perfiles para decidir la limpieza.

La referencia histórica de tu importación es **48 247 filas y 90 columnas**. El notebook verifica los valores reales; no da por hecho que coincidan. No exige todavía `calendar` ni reseñas cargadas en PostgreSQL.

## 1. Comprobar entorno y localizar el proyecto

Usamos pandas, SQLAlchemy y el controlador psycopg2 que ya instalaste. No se necesitan PyArrow, Parquet ni pyodbc en este paso. Si falta una dependencia, verifica primero el kernel seleccionado.

In [1]:
import sys
import json
import csv
from pathlib import Path
from datetime import datetime, timezone
from getpass import getpass
from collections import Counter
from importlib.metadata import version

import numpy as np
import pandas as pd
from sqlalchemy import create_engine, text
from sqlalchemy.engine import URL
from IPython.display import display

print("Python del notebook:", sys.executable)
print("Versión:", sys.version.split()[0])
for paquete in ["pandas", "numpy", "SQLAlchemy", "psycopg2-binary"]:
    print(f"{paquete}: {version(paquete)}")

# Solo si la detección automática falla, escribe la ruta de la carpeta principal:
# RAIZ_MANUAL = r"C:\Users\user\...\Proy_Analysis_Airbnb_Business_Model"
RAIZ_MANUAL = ""
if RAIZ_MANUAL:
    RAIZ = Path(RAIZ_MANUAL).expanduser().resolve()
else:
    ubicacion = Path.cwd().resolve()
    candidatos = [ubicacion, *ubicacion.parents]
    RAIZ = next((p for p in candidatos
                 if (p / "data" / "raw").is_dir() and (p / "src").is_dir()), None)
if RAIZ is None or not (RAIZ / "data" / "raw").is_dir():
    raise RuntimeError("No se encontró el proyecto. Configura RAIZ_MANUAL y repite la celda.")

RUTA_CSV = RAIZ / "data" / "raw" / "listings.csv"
RUTA_REPORTES = RAIZ / "reports" / "diagnostico"
print("Proyecto:", RAIZ)
print("CSV encontrado:", RUTA_CSV.is_file())
pd.set_option("display.max_columns", 12)
pd.set_option("display.max_rows", 30)

Python del notebook: c:\Users\user\Documents\Inteligencia de negocios\Proyecto_IN\Proy_Analysis_Airbnb_Business_Model\.venv\Scripts\python.exe
Versión: 3.11.9
pandas: 3.1.1
numpy: 2.4.6
SQLAlchemy: 2.1.8
psycopg2-binary: 2.9.13
Proyecto: C:\Users\user\Documents\Inteligencia de negocios\Proyecto_IN\Proy_Analysis_Airbnb_Business_Model
CSV encontrado: True


## 2. Leer PostgreSQL

No ejecutes otra vez `importar_listings.py`: la tabla ya está cargada. Esta lectura usa una transacción de solo lectura y una misma instantánea para los conteos y los datos. La contraseña no se escribe en el notebook.

Si aparece `connection refused`, revisa que PostgreSQL esté iniciado y el puerto sea correcto. Si aparece `password authentication failed`, revisa usuario y contraseña. Si no encuentra la tabla, confirma la base de datos y el esquema `staging`.

In [2]:
HOST = "localhost"
PORT = 5432
DATABASE = "proy_airbnb_rio"
USER = "postgres"

motor = create_engine(
    URL.create("postgresql+psycopg2", username=USER,
               password=getpass(f"Contraseña de {USER}: "),
               host=HOST, port=PORT, database=DATABASE),
    connect_args={"connect_timeout": 10},
)
try:
    with motor.connect() as conexion:
        with conexion.begin():
            conexion.execute(text(
                "SET TRANSACTION ISOLATION LEVEL REPEATABLE READ, READ ONLY"
            ))
            columnas_bd = pd.read_sql_query(text("""
                SELECT column_name, data_type, ordinal_position
                FROM information_schema.columns
                WHERE table_schema = 'staging' AND table_name = 'listings'
                ORDER BY ordinal_position
            """), conexion)
            if columnas_bd.empty:
                raise RuntimeError("No se encontró staging.listings en esta base de datos.")
            total_sql = int(conexion.execute(
                text("SELECT COUNT(*) FROM staging.listings")
            ).scalar_one())
            df = pd.read_sql_query(text("SELECT * FROM staging.listings"), conexion)
finally:
    motor.dispose()
    del motor

if len(df) != total_sql:
    raise RuntimeError("El conteo SQL y la lectura no coinciden; revisa la carga.")
if "id" not in df.columns:
    raise ValueError("Falta la columna id: comprueba que usaste el listings detallado.")
if df.empty:
    raise ValueError("La tabla está vacía. Revisa la importación antes de continuar.")
print(f"Lectura completada: {len(df):,} filas y {df.shape[1]} columnas.")
print("Tipos declarados en PostgreSQL:")
display(columnas_bd["data_type"].value_counts().rename_axis("tipo").to_frame("columnas"))

Lectura completada: 48,247 filas y 90 columnas.
Tipos declarados en PostgreSQL:


,columnas
tipo,
text,90


## 3. Registros e identificadores

Un ID identifica un anuncio, no una persona, reserva ni vivienda única comprobada. Mantenemos los identificadores como texto para evitar pérdida de precisión. Los espacios se normalizan solo en una serie auxiliar para detectar problemas.

In [3]:
def texto_aux(serie):
    return serie.astype("string").str.strip()

def ausente(serie):
    s = texto_aux(serie)
    return s.isna() | s.eq("").fillna(False)

ids = texto_aux(df["id"])
ids_presentes = ids[~ausente(df["id"])]
resumen = {
    "generado_utc": datetime.now(timezone.utc).isoformat(),
    "base_de_datos": DATABASE,
    "tabla": "staging.listings",
    "filas_sql": total_sql,
    "filas_dataframe": len(df),
    "columnas": df.shape[1],
    "columnas_no_text_en_bd": int(columnas_bd["data_type"].ne("text").sum()),
    "id_ausentes": int(ausente(df["id"]).sum()),
    "id_unicos_no_vacios": int(ids_presentes.nunique()),
    "filas_excedentes_por_id_repetido": int(ids_presentes.duplicated().sum()),
    "filas_con_id_repetido": int(ids_presentes.duplicated(keep=False).sum()),
    "id_no_compuesto_solo_de_digitos": int((~ids_presentes.str.fullmatch(r"[0-9]+")).sum()),
    "filas_completamente_duplicadas": int(df.duplicated().sum()),
    "referencia_importacion_anterior": {"filas": 48247, "columnas": 90},
}
display(pd.Series(resumen).drop("referencia_importacion_anterior").to_frame("resultado"))
if len(df) != 48247 or df.shape[1] != 90:
    print("AVISO: difiere de la importación anterior. Confirma el archivo y su fecha de corte.")
vista = [c for c in ["id", "neighbourhood_cleansed", "room_type", "price", "last_scraped"] if c in df]
display(df[vista].head(5))

,resultado
generado_utc,2026-11-21T18:16:13.509594+00:00
base_de_datos,proy_airbnb_rio
tabla,staging.listings
filas_sql,48247
filas_dataframe,48247
columnas,90
columnas_no_text_en_bd,0
id_ausentes,0
id_unicos_no_vacios,48247
filas_excedentes_por_id_repetido,0


,id,neighbourhood_cleansed,room_type,price,last_scraped
0,17878,Copacabana,Entire home/apt,$513.00,2026-09-27
1,25026,Copacabana,Entire home/apt,$422.00,2026-09-27
2,35764,Copacabana,Entire home/apt,$366.00,2026-09-27
3,48305,Ipanema,Entire home/apt,"$3,761.43",2026-09-26
4,48901,Copacabana,Entire home/apt,"$1,173.25",2026-09-27


## 4. Perfil de columnas

Separamos `NULL`, cadenas vacías y espacios. Los textos como `N/A`, `null` o `None` se cuentan aparte como posibles marcadores: no se sustituyen automáticamente. Los porcentajes de ausencia combinan únicamente NULL, vacío y espacios.

In [4]:
filas_perfil = []
for columna in df.columns:
    original = df[columna]
    s = texto_aux(original)
    sin_dato = ausente(original)
    filas_perfil.append({
        "columna": columna,
        "tipo_postgresql": columnas_bd.set_index("column_name").at[columna, "data_type"],
        "nulos_sql": int(original.isna().sum()),
        "vacios_o_solo_espacios": int(s.eq("").fillna(False).sum()),
        "ausentes_total": int(sin_dato.sum()),
        "porcentaje_ausentes": round(float(sin_dato.mean() * 100), 2),
        "posibles_marcadores_textuales": int(s.str.lower().isin(["null", "none", "nan", "n/a", "na"]).sum()),
        "valores_distintos_no_vacios": int(s[~sin_dato].nunique()),
        "espacios_en_extremos": int((original.astype("string").ne(s)).fillna(False).sum()),
    })
perfil_columnas = pd.DataFrame(filas_perfil).sort_values("porcentaje_ausentes", ascending=False)
display(perfil_columnas.head(25))
print("No eliminaremos columnas solo por tener muchos valores ausentes.")

,columna,tipo_postgresql,nulos_sql,vacios_o_solo_espacios,ausentes_total,porcentaje_ausentes,posibles_marcadores_textuales,valores_distintos_no_vacios,espacios_en_extremos
7,neighborhood_overview,text,48247,0,48247,100.00,0,0,0
14,host_since,text,48247,0,48247,100.00,0,0,0
21,host_response_time,text,48247,0,48247,100.00,0,0,0
25,host_thumbnail_url,text,48247,0,48247,100.00,0,0,0
23,host_acceptance_rate,text,48247,0,48247,100.00,0,0,0
22,host_response_rate,text,48247,0,48247,100.00,0,0,0
35,neighbourhood_group_cleansed,text,48247,0,48247,100.00,0,0,0
33,neighbourhood,text,48247,0,48247,100.00,0,0,0
29,host_total_listings_count,text,48247,0,48247,100.00,0,0,0
27,host_neighbourhood,text,48247,0,48247,100.00,0,0,0


No eliminaremos columnas solo por tener muchos valores ausentes.


## 5. Diagnosticar el precio anunciado

Este parser provisional reconoce números con punto decimal y coma de miles: `150.00`, `$1,250.00`, `R$ 150.00`. **No** interpreta automáticamente `1.250,00` ni formatos ambiguos. Los marca para revisión.

Que un valor incluya `$` no demuestra que esté expresado en USD. Antes de publicar métricas, confirma la moneda y unidad en el diccionario de Inside Airbnb del corte descargado. No estamos midiendo ingresos, ocupación ni precios efectivamente pagados. Los extremos se describen; no se eliminan.

In [5]:
def precio_aux(serie):
    s = texto_aux(serie)
    patron = r"(?:R\$\s*|\$\s*)?-?(?:[0-9]+|[0-9]{1,3}(?:,[0-9]{3})+)(?:\.[0-9]{1,2})?"
    reconocido = s.str.fullmatch(patron, na=False)
    limpio = s.where(reconocido).str.replace(r"^(?:R\$|\$)\s*", "", regex=True)
    limpio = limpio.str.replace(",", "", regex=False)
    numero = pd.to_numeric(limpio, errors="coerce")
    return numero, (~ausente(serie) & (~reconocido | numero.isna()))

if "price" not in df.columns:
    raise ValueError("No existe price. Revisa la fuente antes de continuar.")
precios, precios_no_interpretables = precio_aux(df["price"])
precio_positivo = precios.gt(0).fillna(False)
resumen.update({
    "precio_ausentes": int(ausente(df["price"]).sum()),
    "precio_no_interpretables": int(precios_no_interpretables.sum()),
    "precio_cero_o_negativo": int(precios.le(0).fillna(False).sum()),
    "precio_positivos_interpretables": int(precio_positivo.sum()),
})
display(pd.Series({k: v for k, v in resumen.items() if k.startswith("precio_")}).to_frame("registros"))
perfil_precios = precios[precio_positivo].describe(percentiles=[.01, .25, .5, .75, .95, .99]).to_frame("precio_anunciado")
display(perfil_precios)
if precios_no_interpretables.any():
    print("Ejemplos de precios no interpretables (revisar formato, no eliminar):")
    display(df.loc[precios_no_interpretables, ["id", "price"]].head(10))

,registros
precio_ausentes,4100
precio_no_interpretables,0
precio_cero_o_negativo,0
precio_positivos_interpretables,44147


,precio_anunciado
count,44147.0
mean,860.69908
std,4434.603391
min,18.97
1%,90.0
25%,300.0
50%,450.0
75%,750.0
95%,2175.0
99%,7553.7746


## 6. Formatos numéricos, rangos básicos y fechas

Se revisan solo columnas presentes. Los rangos de disponibilidad y valoraciones son alertas de coherencia, no reglas definitivas de descarte. Las coordenadas se comprueban contra los límites globales; su pertenencia a Río se validará después con la geografía. La fecha de raspado no representa una serie histórica de precios.

In [6]:
limites = {
    "accommodates": (1, None), "bedrooms": (0, None), "beds": (0, None),
    "bathrooms": (0, None), "minimum_nights": (1, None),
    "maximum_nights": (1, None), "number_of_reviews": (0, None),
    "reviews_per_month": (0, None), "review_scores_rating": (0, 5),
    "availability_30": (0, 30), "availability_60": (0, 60),
    "availability_90": (0, 90), "availability_365": (0, 365),
    "latitude": (-90, 90), "longitude": (-180, 180),
}
filas_numericas = []
for c, (minimo, maximo) in limites.items():
    if c not in df:
        continue
    s = texto_aux(df[c])
    convertido = pd.to_numeric(s, errors="coerce")
    finito = convertido.notna() & np.isfinite(convertido.fillna(0).astype(float))
    numero = convertido.where(finito)
    fuera = numero.lt(minimo).fillna(False)
    if maximo is not None:
        fuera = fuera | numero.gt(maximo).fillna(False)
    filas_numericas.append({
        "columna": c, "ausentes": int(ausente(df[c]).sum()),
        "formato_invalido_o_no_finito": int((~ausente(df[c]) & ~finito).sum()),
        "fuera_rango_referencia": int(fuera.sum()),
        "min_observado": numero.min(), "mediana": numero.median(),
        "max_observado": numero.max(),
    })
perfil_numerico = pd.DataFrame(filas_numericas)
display(perfil_numerico)

filas_fechas = []
for c in ["last_scraped", "host_since", "first_review", "last_review", "calendar_last_scraped"]:
    if c not in df:
        continue
    fechas = pd.to_datetime(texto_aux(df[c]), format="%Y-%m-%d", errors="coerce")
    filas_fechas.append({
        "columna": c, "ausentes": int(ausente(df[c]).sum()),
        "formato_invalido": int((~ausente(df[c]) & fechas.isna()).sum()),
        "min_fecha": str(fechas.min()), "max_fecha": str(fechas.max()),
    })
perfil_fechas = pd.DataFrame(filas_fechas)
display(perfil_fechas)

,columna,ausentes,formato_invalido_o_no_finito,fuera_rango_referencia,min_observado,mediana,max_observado
0,accommodates,0,0,0,1,4.0,16
1,bedrooms,22,0,0,0,1.0,32
2,beds,368,0,0,1,2.0,52
3,bathrooms,48247,0,0,<NA>,<NA>,<NA>
4,minimum_nights,281,0,0,1,2.0,730
5,maximum_nights,281,0,0,1,365.0,9999
6,number_of_reviews,0,0,0,0,9.0,814
7,reviews_per_month,9058,0,0,0.01,0.98,24.31
8,review_scores_rating,9058,0,0,1.0,4.92,5.0
9,availability_30,0,0,0,0,21.0,30


,columna,ausentes,formato_invalido,min_fecha,max_fecha
0,last_scraped,0,0,2026-09-23 00:00:00,2026-09-29 00:00:00
1,host_since,48247,0,NaT,NaT
2,first_review,9058,0,2010-06-07 00:00:00,2026-09-25 00:00:00
3,last_review,9058,0,2012-02-21 00:00:00,2026-09-29 00:00:00
4,calendar_last_scraped,0,0,2026-09-23 00:00:00,2026-09-29 00:00:00


## 7. Cobertura geográfica y tipos de alojamiento

Un valor vacío se muestra con una etiqueta únicamente en estos resúmenes. No se cambia el DataFrame de origen.

In [7]:
categorias = []
for c in ["neighbourhood_group_cleansed", "neighbourhood_cleansed", "room_type", "property_type"]:
    if c not in df:
        continue
    s = texto_aux(df[c]).mask(ausente(df[c]), "<AUSENTE>")
    conteo = s.value_counts(dropna=False).rename_axis("valor").reset_index(name="anuncios")
    conteo.insert(0, "columna", c)
    conteo["porcentaje"] = (100 * conteo["anuncios"] / len(df)).round(2)
    categorias.append(conteo)
    print(c)
    display(conteo.head(10))
perfil_categorias = pd.concat(categorias, ignore_index=True) if categorias else pd.DataFrame()

neighbourhood_group_cleansed


,columna,valor,anuncios,porcentaje
0,neighbourhood_group_cleansed,<AUSENTE>,48247,100.0


neighbourhood_cleansed


,columna,valor,anuncios,porcentaje
0,neighbourhood_cleansed,Copacabana,14938,30.96
1,neighbourhood_cleansed,Ipanema,3887,8.06
2,neighbourhood_cleansed,Barra da Tijuca,3709,7.69
3,neighbourhood_cleansed,Centro,3282,6.8
4,neighbourhood_cleansed,Recreio dos Bandeirantes,2511,5.2
5,neighbourhood_cleansed,Jacarepaguá,2081,4.31
6,neighbourhood_cleansed,Botafogo,2051,4.25
7,neighbourhood_cleansed,Leblon,1931,4.0
8,neighbourhood_cleansed,Santa Teresa,1660,3.44
9,neighbourhood_cleansed,Flamengo,1005,2.08


room_type


,columna,valor,anuncios,porcentaje
0,room_type,Entire home/apt,39334,81.53
1,room_type,Private room,8310,17.22
2,room_type,Shared room,582,1.21
3,room_type,Hotel room,21,0.04


property_type


,columna,valor,anuncios,porcentaje
0,property_type,Entire rental unit,32862,68.11
1,property_type,Private room in rental unit,4834,10.02
2,property_type,Entire home,2336,4.84
3,property_type,Entire condo,1767,3.66
4,property_type,Private room in home,1516,3.14
5,property_type,Entire loft,868,1.8
6,property_type,Entire serviced apartment,695,1.44
7,property_type,Room in hotel,563,1.17
8,property_type,Private room in bed and breakfast,519,1.08
9,property_type,Private room in condo,339,0.7


## 8. Comparación del CSV con PostgreSQL

Este control lee el CSV con un lector que respeta campos entrecomillados y saltos de línea dentro del texto. No cuenta líneas físicas ni filas mostradas por Excel. Compara encabezados, cantidad de registros y multiconjunto de IDs; **no certifica igualdad de todos los demás campos**.

Si el archivo no está en `data/raw/listings.csv`, corrige `RUTA_CSV` en la celda 1. Si sus cifras no coinciden con PostgreSQL, revisaremos la causa antes de limpiar.

In [8]:
comparacion_csv = {"archivo_encontrado": RUTA_CSV.is_file()}
if RUTA_CSV.is_file():
    # Aumentar el límite de campos de csv, compatible con Windows.
    limite_campo = sys.maxsize
    while True:
        try:
            csv.field_size_limit(limite_campo)
            break
        except OverflowError:
            limite_campo //= 10
    ids_csv = Counter()
    registros_csv = 0
    filas_con_ancho_distinto = 0
    with RUTA_CSV.open("r", encoding="utf-8-sig", newline="") as entrada:
        lector = csv.reader(entrada, strict=True)
        encabezados_csv = next(lector, [])
        if "id" not in encabezados_csv:
            raise ValueError("El CSV no tiene encabezado id.")
        posicion_id = encabezados_csv.index("id")
        for fila in lector:
            if not fila:  # Línea vacía física fuera de un registro.
                continue
            registros_csv += 1
            if len(fila) != len(encabezados_csv):
                filas_con_ancho_distinto += 1
            if len(fila) > posicion_id and fila[posicion_id].strip():
                ids_csv[fila[posicion_id].strip()] += 1
    comparacion_csv.update({
        "registros_csv": registros_csv,
        "columnas_csv": len(encabezados_csv),
        "filas_con_numero_de_campos_incorrecto": filas_con_ancho_distinto,
        "mismos_encabezados_y_orden": encabezados_csv == list(df.columns),
        "mismo_numero_de_registros": registros_csv == len(df),
        "mismos_ids_y_frecuencias": ids_csv == Counter(ids_presentes.tolist()),
    })
else:
    print("No se comparó el CSV. Corrige la ruta para completar este control.")
resumen["comparacion_csv"] = comparacion_csv
display(pd.Series(comparacion_csv).to_frame("resultado"))

,resultado
archivo_encontrado,True
registros_csv,48247
columnas_csv,90
filas_con_numero_de_campos_incorrecto,0
mismos_encabezados_y_orden,True
mismo_numero_de_registros,True
mismos_ids_y_frecuencias,True


## 9. Guardar evidencia del diagnóstico

Los informes contienen conteos y perfiles agregados, sin contraseñas ni descripciones o nombres de anfitriones. Si ejecutas otra vez esta celda, reemplazará estos informes de diagnóstico. La tabla de PostgreSQL permanece intacta.

In [9]:
RUTA_REPORTES.mkdir(parents=True, exist_ok=True)
informes = {
    "perfil_columnas.csv": perfil_columnas,
    "perfil_precios.csv": perfil_precios.reset_index(names="estadistico"),
    "perfil_numerico.csv": perfil_numerico,
    "perfil_fechas.csv": perfil_fechas,
    "perfil_categorias.csv": perfil_categorias,
}
for nombre, tabla in informes.items():
    tabla.to_csv(RUTA_REPORTES / nombre, index=False, encoding="utf-8-sig")
archivo_resumen = RUTA_REPORTES / "resumen_diagnostico.json"
archivo_resumen.write_text(json.dumps(resumen, ensure_ascii=False, indent=2), encoding="utf-8")
print("Diagnóstico completado. Informes en:", RUTA_REPORTES)
print(json.dumps(resumen, ensure_ascii=False, indent=2))

Diagnóstico completado. Informes en: C:\Users\user\Documents\Inteligencia de negocios\Proyecto_IN\Proy_Analysis_Airbnb_Business_Model\reports\diagnostico
{
  "generado_utc": "2026-11-21T18:16:13.509594+00:00",
  "base_de_datos": "proy_airbnb_rio",
  "tabla": "staging.listings",
  "filas_sql": 48247,
  "filas_dataframe": 48247,
  "columnas": 90,
  "columnas_no_text_en_bd": 0,
  "id_ausentes": 0,
  "id_unicos_no_vacios": 48247,
  "filas_excedentes_por_id_repetido": 0,
  "filas_con_id_repetido": 0,
  "id_no_compuesto_solo_de_digitos": 0,
  "filas_completamente_duplicadas": 0,
  "referencia_importacion_anterior": {
    "filas": 48247,
    "columnas": 90
  },
  "precio_ausentes": 4100,
  "precio_no_interpretables": 0,
  "precio_cero_o_negativo": 0,
  "precio_positivos_interpretables": 44147,
  "comparacion_csv": {
    "archivo_encontrado": true,
    "registros_csv": 48247,
    "columnas_csv": 90,
    "filas_con_numero_de_campos_incorrecto": 0,
    "mismos_encabezados_y_orden": true,
    "mi

## Qué revisaremos antes de la fase 1.2

- Que el conteo de PostgreSQL coincida con la lectura y el CSV de origen, y que el identificador no falte ni se repita sin explicación.
- Ausencias, formatos no interpretables y rangos que requieren comprobación, especialmente `price`, capacidad y ubicación.
- El corte y la moneda documentados por Inside Airbnb. El símbolo `$` por sí solo no basta.
- Qué columnas necesita el estudio de oferta y cuáles el modelo predictivo.

**Siguiente entrega:** reglas de limpieza justificadas, datos tipados en una capa de trabajo y registro de calidad. Mantendremos anuncios sin precio cuando sirvan para analizar la oferta; su utilidad para entrenar un modelo supervisado se decidirá aparte. No imputaremos el precio objetivo ni borraremos extremos automáticamente.

En este punto comparte `resumen_diagnostico.json`, `perfil_columnas.csv` y `perfil_numerico.csv`. Si falla una celda, comparte el mensaje de error sin credenciales.

Documentación técnica: [pandas.read_sql_query](https://pandas.pydata.org/docs/reference/api/pandas.read_sql_query.html), [PostgreSQL SET TRANSACTION](https://www.postgresql.org/docs/17/sql-set-transaction.html).

**Estado del archivo entregado:** preparado y revisado con datos sintéticos. Sus resultados reales se generarán al ejecutarlo contra tu PostgreSQL local.